<a href="https://colab.research.google.com/github/Tasnym001/AbbaBashir93/blob/main/GUI_for_MLP_FA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install mealpy

In [2]:
pip install deap

In [3]:
pip install --upgrade mealpy

  Using cached mealpy-3.0.3-py3-none-any.whl.metadata (168 kB)
Using cached mealpy-3.0.3-py3-none-any.whl (423 kB)
  Attempting uninstall: mealpy
    Found existing installation: mealpy 2.4.2
    Uninstalling mealpy-2.4.2:
      Successfully uninstalled mealpy-2.4.2


In [4]:
pip install mealpy==2.4.2

  Using cached mealpy-2.4.2-py3-none-any.whl.metadata (57 kB)
Using cached mealpy-2.4.2-py3-none-any.whl (343 kB)
  Attempting uninstall: mealpy
    Found existing installation: mealpy 3.0.3
    Uninstalling mealpy-3.0.3:
      Successfully uninstalled mealpy-3.0.3


In [5]:
import mealpy
print(mealpy.__version__)

2.4.2


In [6]:
import mealpy.swarm_based
print(dir(mealpy.swarm_based))

['ABC', 'ACOR', 'ALO', 'AO', 'BA', 'BES', 'BFO', 'BSA', 'BeesA', 'COA', 'CSA', 'CSO', 'DO', 'EHO', 'FA', 'FFA', 'FOA', 'GOA', 'GWO', 'HGS', 'HHO', 'JA', 'MFO', 'MRFO', 'MSA', 'NMRA', 'PFA', 'PSO', 'SFO', 'SHO', 'SLO', 'SRSR', 'SSA', 'SSO', 'SSpiderA', 'SSpiderO', 'WOA', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__']


In [7]:
import mealpy
print(dir(mealpy.bio_based))

['BBO', 'EOA', 'IWO', 'SBO', 'SMA', 'TPO', 'VCS', 'WHO', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__']


In [8]:
# ==============================
# Install dependencies (if needed)
# ==============================
!pip install ipywidgets mealpy scikit-learn pandas

# ==============================
# Import libraries
# ==============================
import pandas as pd
import numpy as np
import ipywidgets as widgets
from IPython.display import display
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from mealpy.swarm_based import FA

# ==============================
# Load your dataset
# ==============================
file_path = "/content/3D CONCRETE RD.xlsx"  # Replace with your actual dataset path
data = pd.read_excel(file_path)

feature_cols = [
    'C (kg/m³)', 'W (kg/m³)', 'S (kg/m³)', 'W/B', 'FA (kg/m³)',
    'SP (kg/m³)', 'LF (ft)', 'SF (kg/m³)', 'VF', 'HMC (kg/m³)',
    'DF (kg/m³)', 'T (mm)', 'As (mm²) ', 'SG', 'Ft (MPa)', 'Li (kg/m³)'
]
target_col = 'CS (MPa)'

X = data[feature_cols]
y = data[target_col]

# Scale
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

# ==============================
# Objective function for FA
# ==============================
def objective_function_gui(params):
    hidden_layer_size_obj = int(params[0])
    learning_rate_obj = params[1]
    model = MLPRegressor(
        hidden_layer_sizes=(hidden_layer_size_obj,),
        learning_rate_init=learning_rate_obj,
        max_iter=500,
        random_state=42
    )
    model.fit(X_train, y_train.values.ravel())
    return mean_squared_error(y_test, model.predict(X_test))

# Optimization
print("Running Firefly Algorithm optimization...")
problem_gui = {
    "fit_func": objective_function_gui,
    "lb": [10, 0.0001],
    "ub": [200, 0.1],
    "minmax": "min",
}
fa_model = FA.BaseFA(problem_gui, epoch=50, pop_size=20, max_sparks=5)
best_params, best_fitness = fa_model.solve()

final_mlp_fa_model = MLPRegressor(
    hidden_layer_sizes=(int(best_params[0]),),
    learning_rate_init=best_params[1],
    max_iter=500,
    random_state=42
)
final_mlp_fa_model.fit(X_train, y_train.values.ravel())

print(f"Best Parameters: Hidden Layer Size = {int(best_params[0])}, Learning Rate = {best_params[1]:.6f}")
print(f"Best MSE: {best_fitness:.4f}")

# ==============================
# Input widgets
# ==============================
input_widgets = {}
for column in feature_cols:
    min_value = X[column].min()
    max_value = X[column].max()
    input_widgets[column] = widgets.FloatText(
        value=float(X[column].mean()),
        description=f"<b>{column}</b> ({min_value:.2f}-{max_value:.2f})",
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='70%', height='30px')
    )

# Output label
result_label = widgets.HTML(
    value="<b>Output:</b>",
    layout=widgets.Layout(width='100%', padding="10px")
)

# Prediction function
def predict_strength(btn):
    inputs = [widget.value for widget in input_widgets.values()]
    inputs_scaled = scaler.transform([inputs])
    prediction = final_mlp_fa_model.predict(inputs_scaled)[0]
    result_label.value = (
        f"<b>3DPC Predicted Compressive Strength:</b> "
        f"<span style='color:blue; font-size: 24px;'>{prediction:.2f} MPa</span>"
    )

# Predict button (RED)
predict_button = widgets.Button(
    description="Calculate",
    layout=widgets.Layout(width='50%', height='40px'),
    style={'button_color': 'red', 'font_weight': 'bold'}
)
predict_button.on_click(predict_strength)

# Input form grid
input_parameters_box = widgets.GridBox(
    children=list(input_widgets.values()),
    layout=widgets.Layout(
        grid_template_columns="repeat(2, 50%)",
        grid_gap="20px 20px",
        width='100%'
    )
)

# Output section
output_box = widgets.VBox([
    widgets.HTML("<h2 style='color:blue; font-size: 22px;'>Target</h2>"),
    result_label
])

# ==============================
# Main container with light ash & yellow theme
# ==============================
header_box = widgets.VBox([
    widgets.HTML("<h1 style='text-align:center; font-size: 28px;'>GUI Model for Predicting Compressive Strength of 3D Printed Concrete</h1>"),
    widgets.HTML("<h3 style='text-align:center; color:red;'>Developed by: Abba Bashir, Sani I. Abba </h3>")
], layout=widgets.Layout(
    background_color="#fff9cc",  # light yellow
    padding='10px',
    border_radius='8px'
))

main_box = widgets.VBox([
    header_box,
    widgets.HTML("<hr>"),
    input_parameters_box,
    widgets.HTML("<br>"),
    predict_button,
    output_box
], layout=widgets.Layout(
    background_color="#f2f2f2",  # light ash
    border='2px solid #ccc',
    padding='20px',
    border_radius='10px',
    margin='auto',
    width='60%',
    box_shadow='5px 5px 10px #888888'
))

# Display
display(main_box)


2025/08/19 05:36:33 PM, INFO, mealpy.swarm_based.FA.BaseFA: Solving single objective optimization problem.
INFO:mealpy.swarm_based.FA.BaseFA:Solving single objective optimization problem.


Running Firefly Algorithm optimization...


/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptro

Best Parameters: Hidden Layer Size = 105, Learning Rate = 0.086547
Best MSE: 71.5980


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(
